In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Fine-tuning a Pre-trained Model for Protein Abundance Prediction
This notebook demonstrates how to fine-tune a pre-trained scPMMI model for surface protein (ADT / antibody-derived tag) prediction from RNA measurements. The task learns to map RNA expression to cell surface protein abundance using paired RNA-ADT (CITE-seq) data.

We summarize the fine-tuning pipeline in the following steps, which can be used as general recipe for fine-tuning on protein abundance prediction task and beyond:

    1. Specify hyper-parameter setup for protein abundance prediction
    2. Load and pre-process data
    3. Load the pre-trained scPMMI model
    4. fine-tune scPMMI with task-specific objectives
    5. Evaluate fine-tuned scPMMI

In [11]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import muon as mu
import scanpy as sc
import muon as mu
import random
from tqdm import tqdm
from typing import Tuple
from scipy.sparse import issparse
import swanlab
import matplotlib.pyplot as plt

import torch
from torch import optim, Tensor, nn
from torch.utils.data import DataLoader, Dataset
from torch.amp.autocast_mode import autocast
from transformers.optimization import get_scheduler

from scpmmi.utils.unify import align_to_common_genes, align_to_common_proteins
from scpmmi.tokenizer.vocab import Vocab
from scpmmi.models.scPMMI import scPMMI
from scpmmi.models.pipeline import ProteinPrediction 
from scpmmi.loss import ContrastiveLoss
from scpmmi.utils.util import set_seed
from scpmmi import logger

## Step 1: Configure hyperparameters for protein prediction

In [12]:
hyperparameter_defaults = dict(
    task='protein_prediction',
    seed=42,
    dataset='cite_bmmc_s2',
    load_model = '../ckpt/ckpt_protra',
    gene_vocab='../config_file/default_gene_vocab.json',
    protein_vocab='../config_file/default_protein_vocab.json',
    batch_size=64,
    num_workers=4,
    temperature=0.05,
    alpha=1.5,
    n_epochs=10,
    lr=1.5e-4,
    weight_decay=0.05,
)

In [13]:
run = swanlab.init(
    project="scPMMI-fine-tuning",
    experiment_name=f"fine-tuning-{time.strftime('%b%d-%H-%M')}",
    config=hyperparameter_defaults,
    reinit=True,
    mode="offline"
)
config = run.config
print(config)
set_seed(config.seed)

swanlab:  ☁️ Run `swanlab sync 
/mnt/disk2/zyuan/code/scPMMI/tutorials/swanlog/run-20260125_151937-or4slm1o8ye5i5gl8og5w` to sync logs to remote 
server

swanlab: The environment variable SWANLAB_MODE will be overwritten by the parameter mode

swanlab: Tracking run with swanlab version 0.6.8

swanlab: Run data will be saved locally in 
/mnt/disk2/zyuan/code/scPMMI/tutorials/swanlog/run-20260125_152122-iiuc039tex8fsv871yzoy

swanlab: Backing up run fine-tuning-Jan25-15-21 locally

swanlab:  ☁️ Run `swanlab sync 
/mnt/disk2/zyuan/code/scPMMI/tutorials/swanlog/run-20260125_152122-iiuc039tex8fsv871yzoy` to sync logs to remote 
server

{'task': 'protein_prediction', 'seed': 42, 'dataset': 'cite_bmmc_s2', 'load_model': '../ckpt/ckpt_protra', 'gene_vocab': '../config_file/default_gene_vocab.json', 'protein_vocab': '../config_file/default_protein_vocab.json', 'batch_size': 64, 'num_workers': 4, 'temperature': 0.05, 'alpha': 1.5, 'n_epochs': 10, 'lr': 0.00015, 'weight_decay': 0.05}


In [14]:
dataset_name = config.dataset
save_path = Path(f"./result/protein_abundance_prediction/dev_{dataset_name}-{time.strftime('%b%d-%H-%M')}/")
save_path.mkdir(parents=True, exist_ok=True)
print(f"save to {save_path}")

save to result/protein_abundance_prediction/dev_cite_bmmc_s2-Jan25-15-21


## Step 2: Load and preprocess data
This section loads paired RNA-ADT (CITE-seq) datasets from `.h5mu` files, constructs train/validation/test splits, and aligns features to the pre-trained model's vocabulary. The code branches handle several prepared datasets (CITE BMMC samples, PBMC, etc.), each with different batch structures.

Key actions:
- Extract `rna` and `adt` (antibody-derived tag) modalities from MuData objects.
- Mark `istrain` flags to designate training vs. test data; validation is split from training data during dataset creation.
- Apply modality-specific preprocessing (log normalization with different scaling constants for ADT vs. RNA).

### 2.1 Load raw RNA-ADT data and construct splits
This subsection contains dataset-specific code that reads `.h5mu` files for different CITE-seq or PBMC studies and constructs training/test partitions. Each dataset branch defines its own `istrain` labels based on batch identifiers or donor IDs.

Best practices:
- Verify that gene and protein names match across the RNA and ADT modalities after loading.
- Check for missing batches or unexpected data shapes before proceeding.
- The code uses different normalization schemes for ADT vs. RNA (1e3 vs. 1e4 scaling), which is intentional for ADT data characteristics.

In [15]:
if config.dataset == "cite_bmmc_s2":
    mdata = mu.read_h5mu(Path("../data/mdata/rna_adt/CITE_BMMC_raw.h5mu"))
    mdata = mdata[mdata.obs['batch'].isin(['s2d1', 's2d4', 's2d5'])].copy()
    adata_rna = mdata.mod['rna'] # type: ignore
    adata_adt = mdata.mod['adt'] # type: ignore
    adata_rna.obs['istrain'] = 'test'
    adata_adt.obs['istrain'] = 'test'
    adata_rna.obs.loc[adata_rna.obs['batch'].isin(['s2d1']), 'istrain'] = 'train'
    adata_adt.obs.loc[adata_adt.obs['batch'].isin(['s2d1']), 'istrain'] = 'train'     

elif config.dataset == "pbmc":
    mdata = mu.read_h5mu(Path("../data/mdata/rna_adt/CITE_PBMC.h5mu"))

    adata_rna = mdata.mod['rna'] # type: ignore
    adata_adt = mdata.mod['adt'] # type: ignore
    adata_rna.obs['cell_type'] = adata_rna.obs['celltype.l1'].values
    adata_rna.obs['istrain'] = 'test'
    adata_adt.obs['istrain'] = 'test'
    train_bool = [x in ['P1', 'P3', 'P4', 'P7'] for x in adata_rna.obs['donor']]
    adata_rna.obs.loc[train_bool, 'istrain'] = 'train'
    adata_adt.obs.loc[train_bool, 'istrain'] = 'train' 

else:
    raise ValueError(f"Unknown dataset: {config.dataset}")

adata_rna, adata_adt

(AnnData object with n_obs × n_vars = 25171 × 13953
     obs: 'GEX_n_genes_by_counts', 'GEX_pct_counts_mt', 'GEX_size_factors', 'GEX_phase', 'ADT_n_antibodies_by_counts', 'ADT_total_counts', 'ADT_iso_count', 'cell_type', 'batch', 'ADT_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker', 'is_train', 'istrain'
     var: 'feature_types', 'gene_id'
     uns: 'dataset_id', 'genome', 'organism'
     obsm: 'ADT_X_pca', 'ADT_X_umap', 'ADT_isotype_controls', 'GEX_X_pca', 'GEX_X_umap'
     layers: 'counts',
 AnnData object with n_obs × n_vars = 25171 × 134
     obs: 'GEX_n_genes_by_counts', 'GEX_pct_counts_mt', 'GEX_size_factors', 'GEX_phase', 'ADT_n_antibodies_by_counts', 'ADT_total_counts', 'ADT_iso_count', 'cell_type', 'batch', 'ADT_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', '

### 2.2 Align genes and proteins to the pre-trained vocabulary
Load the pre-trained `Vocab` files for genes and proteins and use alignment utilities to reorder your dataset features to match the model's expected token order. Missing proteins are typically filled with zeros; ensure your downstream preprocessing handles this correctly.

In [16]:
gene_vocab = Vocab.from_file(config.gene_vocab)
gene_list = gene_vocab.get_token_list()
gene_ids = gene_vocab(gene_list)
adata_rna = align_to_common_genes(adata_rna, gene_list)    

2026-01-25 15:21:26 - scPMMI - INFO - Number of matched genes: 12316


In [17]:
protein_vocab = Vocab.from_file(config.protein_vocab)
protein_list = protein_vocab.get_token_list()
protein_ids = protein_vocab(protein_list)
adata_adt = align_to_common_proteins(adata_adt, protein_list)

2026-01-25 15:21:49 - scPMMI - INFO - Number of matched proteins: 134


## Step 3: Load the pre-trained scPMMI model
Load the model architecture from the checkpoint's `config.json` and initialize weights from `model.bin`. The notebook uses the `ProteinPrediction` pipeline class which wraps the encoder and adds protein prediction heads.

Key behaviors:
- Attempt a full state dict load; if keys or shapes mismatch, fall back to safe partial loading (only matching parameters).
- Move the model to the target CUDA device and set the numeric dtype (here: `torch.bfloat16` for memory efficiency).
- The `ProteinPrediction` class includes both contrastive and reconstruction objectives; ensure `do_mvc=True` is set for multi-view prediction heads.

In [18]:
device = 'cuda:1'
torch.cuda.set_device(device)
dtype = torch.bfloat16
model_dir = Path(config.load_model)
model_config_file = model_dir / "config.json"
model_ckpt = model_dir / "model.bin"
with open(model_config_file, "r") as f:
    model_config = json.load(f)
model = ProteinPrediction(model_config, do_mvc=True)

if config.load_model is not None:
    try:
        model.load_state_dict(torch.load(model_ckpt))
        logger.info(f"Loading all model params from {model_ckpt}")
    except:
        # only load params that are in the model and match the size
        model_dict = model.state_dict()
        pretrained_dict = torch.load(model_ckpt)
        pretrained_dict = {
            k: v
            for k, v in pretrained_dict.items()
            if k in model_dict and v.shape == model_dict[k].shape
        }
        for k, v in pretrained_dict.items():
            logger.info(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)
model = model.to(device, dtype)

2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.embedding.cls_token with shape torch.Size([1, 1, 512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.embedding.patch_position_embeddings with shape torch.Size([1, 160, 512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.embedding.embedding.weight with shape torch.Size([40957, 512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.layers.0.norm.weight with shape torch.Size([512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.layers.0.mixer.Wqkv.weight with shape torch.Size([1536, 512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.layers.0.mixer.Wqkv.bias with shape torch.Size([1536])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params encoder_mod1.backbone.layers.0.mixer.out_proj.weight with shape torch.Size([512, 512])
2026-01-25 15:21:52 - scPMMI - INFO - Loading params

## Step 4: Fine-tune the model with cross-modal reconstruction objectives

In [23]:
class MultiSeqDataset(Dataset):
    def __init__(
        self,
        adata_rna: sc.AnnData,
        feature_ids_mod1: list,
        adata_adt: sc.AnnData,
        feature_ids_mod2: list,
    ):
        """
        PyTorch-compatible dataset for paired multi-modal single-cell sequencing data.

        Args:
            adata_mod1: AnnData object for the first modality (e.g., RNA).
            layer_mod1: Layer name in adata_mod1 to use as input.
            adata_mod2: AnnData object for the second modality (e.g., ATAC).
            layer_mod2: Layer name in adata_mod2 to use as input.
        """
        self.gene_ids = torch.tensor(feature_ids_mod1, dtype=torch.int32)
        self.protein_ids = torch.tensor(feature_ids_mod2, dtype=torch.int32)
        # ---------------------- Process RNA modality ----------------------
        X_rna = adata_rna.X.toarray() if issparse(adata_rna.X) else adata_rna.X # type: ignore
        # Normalize and log-transform
        X_rna_input = np.log1p(X_rna / X_rna.sum(1)[:, None] * 1e4)
        X_rna_input = torch.tensor(X_rna_input)
        gene_ids = self.gene_ids.unsqueeze(0).repeat(X_rna_input.size(0), 1)

        # ---------------------- Process ADT modality ----------------------
        X_adt = adata_adt.X.toarray() if issparse(adata_adt.X) else adata_adt.X # type: ignore

        # Normalize and log-transform
        X_adt_input = np.log1p(X_adt / np.maximum(X_adt.sum(1)[:, None], 1) * 1e3)

        X_adt_input = torch.tensor(X_adt_input)

        protein_ids = self.protein_ids.unsqueeze(0).repeat(X_adt_input.size(0), 1)
        
        # ---------------------- Store both modalities ----------------------
        self.data = {
            "mod1_token_ids": gene_ids,
            "mod1_values": X_rna_input,
            "mod1_values_target": X_rna_input,

            "mod2_token_ids": protein_ids,
            "mod2_values": X_adt_input,
            "mod2_values_target": X_adt_input,
        }

    def __len__(self):
        return self.data["mod1_token_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}

In [24]:
adata_rna_train_raw = adata_rna[adata_rna.obs['istrain'] == 'train']
adata_rna_test = adata_rna[adata_rna.obs['istrain'] != 'train']
adata_adt_train_raw = adata_adt[adata_adt.obs['istrain'] == 'train']
adata_adt_test = adata_adt[adata_adt.obs['istrain'] != 'train']

In [25]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(adata_rna_train_raw.n_obs),
    test_size=0.1,
    stratify=adata_rna_train_raw.obs["cell_type"]
)

adata_rna_train = adata_rna_train_raw[train_idx].copy()
adata_rna_val  = adata_rna_train_raw[val_idx].copy()

adata_adt_train = adata_adt_train_raw[train_idx].copy()
adata_adt_val  = adata_adt_train_raw[val_idx].copy()

In [26]:

train_dataset = MultiSeqDataset(adata_rna=adata_rna_train, feature_ids_mod1=gene_ids, 
                            adata_adt=adata_adt_train, feature_ids_mod2=protein_ids)
val_dataset = MultiSeqDataset(adata_rna=adata_rna_val, feature_ids_mod1=gene_ids, 
                            adata_adt=adata_adt_val, feature_ids_mod2=protein_ids)
test_dataset = MultiSeqDataset(adata_rna=adata_rna_test, feature_ids_mod1=gene_ids, 
                            adata_adt=adata_adt_test, feature_ids_mod2=protein_ids)


train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=False,
                    num_workers=config.num_workers, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                    num_workers=config.num_workers, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False,
                    num_workers=config.num_workers, pin_memory=True)

In [27]:
criterion_clip = ContrastiveLoss(
    temperature=config.temperature, alpha=config.alpha
)
criterion_mse1 = nn.MSELoss()
criterion_mse2 = nn.MSELoss()
optimizer = optim.AdamW( # type: ignore
    model.parameters(), lr=config.lr, weight_decay=config.weight_decay
)
max_train_steps = config.n_epochs * len(train_dataloader)
num_warmup_steps = int(0.1 * max_train_steps)  # 10% steps for warmup
scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=max_train_steps,
)

In [28]:
def evaluate(model: nn.Module, loader: DataLoader) -> Tuple[float, float, float, float, float]:
    """
    Evaluate the model on the evaluation data.
    """
    model.eval()
    total_mse2t2_loss = 0.0
    total_mse1t2_loss = 0.0
    total_mse1_loss = 0.0
    total_mse2_loss = 0.0
    total_loss = 0.0
    with torch.no_grad():
        for batch_data in loader:
            mod1_token_ids = batch_data["mod1_token_ids"].to(device)
            mod1_values_input = batch_data["mod1_values"].to(device)
            mod1_values_target = batch_data["mod1_values_target"].to(device)

            mod2_token_ids = batch_data["mod2_token_ids"].to(device)
            mod2_values_input = batch_data["mod2_values"].to(device)
            mod2_values_target = batch_data["mod2_values_target"].to(device)

            with autocast(device_type="cuda", dtype=dtype):
                output = model(mod1_token_ids, mod1_values_input, mod2_token_ids, mod2_values_input, MVC=True)
                loss_clip = criterion_clip(output['cell_emb_mod1'], output['cell_emb_mod2'])
                
                loss_mse1t2 = criterion_mse1(output['pred_mod1t2'], mod2_values_target.to(output['pred_mod1t2']))
                loss_mse2t2 = criterion_mse2(output['pred_mod2'], mod2_values_target.to(output['pred_mod2']))

                loss_mse1 = criterion_mse1(output['mvc_output_mod1t2'], mod2_values_target.to(output['mvc_output_mod1t2']))
                loss_mse2 = criterion_mse2(output['mvc_output_mod2t2'], mod2_values_target.to(output['mvc_output_mod2t2']))

                loss = loss_clip + loss_mse1t2 + loss_mse2t2 + loss_mse1 + loss_mse2

            total_loss += loss.item()
            total_mse1t2_loss += loss_mse1t2.item()
            total_mse2t2_loss += loss_mse2t2.item()
            total_mse1_loss += loss_mse1.item()
            total_mse2_loss += loss_mse2.item()
    
    return (total_loss / len(loader), 
            total_mse1t2_loss / len(loader), total_mse2t2_loss / len(loader), 
            total_mse1_loss / len(loader), total_mse2_loss / len(loader))

In [29]:
import copy

best_val_loss = float("inf")
global_step = 0
for epoch in range(config.n_epochs):
    model.train()
    loop = tqdm(train_dataloader, desc=f'Epoch [{epoch}/{config.n_epochs}]',leave=False)
    for step, batch_data in enumerate(loop):
        mod1_token_ids = batch_data["mod1_token_ids"].to(device)
        mod1_values_input = batch_data["mod1_values"].to(device)
        mod1_values_target = batch_data["mod1_values_target"].to(device)

        mod2_token_ids = batch_data["mod2_token_ids"].to(device)
        mod2_values_input = batch_data["mod2_values"].to(device)
        mod2_values_target = batch_data["mod2_values_target"].to(device)

        optimizer.zero_grad()
        with autocast(device_type="cuda", dtype=dtype):
            output = model(
                mod1_token_ids, mod1_values_input, 
                mod2_token_ids, mod2_values_input,
                MVC=True,
            )
            metric_to_log = {}
            loss_clip = criterion_clip(output['cell_emb_mod1'], output['cell_emb_mod2'])
            loss = loss_clip
            metric_to_log['clip'] = loss_clip.item()

            loss_mse1t2 = criterion_mse1(output['pred_mod1t2'], mod2_values_target.to(output['pred_mod1t2']))
            loss_mse2t2 = criterion_mse2(output['pred_mod2'], mod2_values_target.to(output['pred_mod2']))
            loss += loss_mse1t2 + loss_mse2t2
            metric_to_log['mse1t2'] = loss_mse1t2.item()
            metric_to_log['mse2t2'] = loss_mse2t2.item()
            

        metric_to_log['loss'] = loss.item()
        metric_to_log['step'] = global_step
        metric_to_log['lr'] = scheduler.get_last_lr()[0]
        loss.backward()
        optimizer.step()
        scheduler.step()
        global_step += 1
        loop.set_postfix(metric_to_log)
                    
        swanlab.log(metric_to_log)
    val_loss, val_mse1t2_loss, val_mse2t2_loss, val_mse1_loss, val_mse2_loss = evaluate(model, val_dataloader)

    logger.info("-" * 80)
    logger.info(
        f"| end of epoch {epoch:3d} | valid loss {val_loss:5.4f} "
        f"valid loss/mse1t2 {val_mse1t2_loss:5.4f} | valid loss {val_mse2t2_loss:5.4f} "
        f"valid loss/mse1 {val_mse1_loss:5.4f} | valid loss/mse2 {val_mse2_loss:5.4f}"
    )
    logger.info("-" * 80)

    if val_mse1t2_loss < best_val_loss:
        best_val_loss = val_mse1t2_loss
        best_model = copy.deepcopy(model)
        best_model_epoch = epoch
        logger.info(f"Best model with score {best_val_loss:5.4f}")

Epoch [0/10]:   0%|          | 0/148 [00:00<?, ?it/s]

2026-01-25 15:22:59 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:22:59 - scPMMI - INFO - | end of epoch   0 | valid loss 5.6819 valid loss/mse1t2 0.1871 | valid loss 0.6772 valid loss/mse1 1.4903 | valid loss/mse2 1.4528
2026-01-25 15:22:59 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:22:59 - scPMMI - INFO - Best model with score 0.1871


2026-01-25 15:23:28 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:23:28 - scPMMI - INFO - | end of epoch   1 | valid loss 5.6724 valid loss/mse1t2 0.1563 | valid loss 0.6740 valid loss/mse1 1.4773 | valid loss/mse2 1.4495
2026-01-25 15:23:29 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:23:29 - scPMMI - INFO - Best model with score 0.1563


2026-01-25 15:23:58 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:23:58 - scPMMI - INFO - | end of epoch   2 | valid loss 5.3925 valid loss/mse1t2 0.1506 | valid loss 0.6759 valid loss/mse1 1.4701 | valid loss/mse2 1.4534
2026-01-25 15:23:58 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:23:58 - scPMMI - INFO - Best model with score 0.1506


2026-01-25 15:24:28 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:24:28 - scPMMI - INFO - | end of epoch   3 | valid loss 5.3010 valid loss/mse1t2 0.1481 | valid loss 0.6704 valid loss/mse1 1.4585 | valid loss/mse2 1.4497
2026-01-25 15:24:28 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:24:28 - scPMMI - INFO - Best model with score 0.1481


2026-01-25 15:24:58 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:24:58 - scPMMI - INFO - | end of epoch   4 | valid loss 5.2473 valid loss/mse1t2 0.1457 | valid loss 0.6667 valid loss/mse1 1.4730 | valid loss/mse2 1.4602
2026-01-25 15:24:58 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:24:58 - scPMMI - INFO - Best model with score 0.1457


2026-01-25 15:25:27 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:25:27 - scPMMI - INFO - | end of epoch   5 | valid loss 5.2361 valid loss/mse1t2 0.1449 | valid loss 0.6662 valid loss/mse1 1.4740 | valid loss/mse2 1.4600
2026-01-25 15:25:27 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:25:28 - scPMMI - INFO - Best model with score 0.1449


2026-01-25 15:25:57 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:25:57 - scPMMI - INFO - | end of epoch   6 | valid loss 5.1935 valid loss/mse1t2 0.1445 | valid loss 0.6642 valid loss/mse1 1.4784 | valid loss/mse2 1.4611
2026-01-25 15:25:57 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:25:57 - scPMMI - INFO - Best model with score 0.1445


2026-01-25 15:26:27 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:26:27 - scPMMI - INFO - | end of epoch   7 | valid loss 5.3161 valid loss/mse1t2 0.1445 | valid loss 0.6642 valid loss/mse1 1.4830 | valid loss/mse2 1.4624
2026-01-25 15:26:27 - scPMMI - INFO - --------------------------------------------------------------------------------


2026-01-25 15:26:57 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:26:57 - scPMMI - INFO - | end of epoch   8 | valid loss 5.3086 valid loss/mse1t2 0.1446 | valid loss 0.6641 valid loss/mse1 1.4835 | valid loss/mse2 1.4656
2026-01-25 15:26:57 - scPMMI - INFO - --------------------------------------------------------------------------------


2026-01-25 15:27:26 - scPMMI - INFO - --------------------------------------------------------------------------------
2026-01-25 15:27:26 - scPMMI - INFO - | end of epoch   9 | valid loss 5.3088 valid loss/mse1t2 0.1446 | valid loss 0.6641 valid loss/mse1 1.4837 | valid loss/mse2 1.4655
2026-01-25 15:27:26 - scPMMI - INFO - --------------------------------------------------------------------------------


In [30]:
best_model.save_pretrained(save_path / 'ckpt')

## Step 5: Inference and Evaluation

In [31]:
best_model.eval()        
with torch.no_grad():
    adt_pred = []
    for batch_data in tqdm(test_dataloader, desc="Inference"):
        mod1_token_ids = batch_data["mod1_token_ids"].to(device)
        mod1_values_input = batch_data["mod1_values"].to(device)
        mod1_values_target = batch_data["mod1_values_target"].to(device)

        mod2_token_ids = batch_data["mod2_token_ids"].to(device)
        mod2_values_input = batch_data["mod2_values"].to(device)
        mod2_values_target = batch_data["mod2_values_target"].to(device)

        output = best_model(mod1_token_ids, mod1_values_input, mod2_token_ids, mod2_values_input, MVC=True)
        
        adt_pred.append(output['pred_mod1t2'].cpu())
    adt_pred = torch.cat(adt_pred, dim=0).to(torch.float32).numpy()
adata_adt_test.layers['adt_pred'] = adt_pred

Inference: 100%|██████████| 230/230 [00:14<00:00, 15.73it/s]


In [37]:
adata_adt_test.layers['counts'] = adata_adt_test.X
sc.pp.filter_genes(adata_adt_test, min_counts=3)
groundtruth = adata_adt_test.X.toarray()
groundtruth = np.log1p(groundtruth / np.maximum(groundtruth.sum(1)[:, None], 1) * 1e3)
pred = adata_adt_test.layers['adt_pred']
adata_adt_test.write(save_path / "protein_test.h5ad")

In [38]:
import numpy as np
from scipy.stats import pearsonr

def evaluate_protein_prediction(y_true, y_pred):

    assert y_true.shape == y_pred.shape
    n_proteins = y_true.shape[1]

    pearson = np.zeros(n_proteins)
    rmse = np.zeros(n_proteins)

    # z-score standardization
    true_z = (y_true - y_true.mean(0)) / y_true.std(0)
    pred_z = (y_pred - y_pred.mean(0)) / y_pred.std(0)

    for i in range(n_proteins):
        # Pearson correlation (ignoring NaNs)
        pearson[i] = pearsonr(y_true[:, i], y_pred[:, i])[0]

        # RMSE (based on z-score standardized features)
        rmse[i] = np.sqrt(np.mean((true_z[:, i] - pred_z[:, i])**2))

    return pearson, rmse


In [39]:
pearson, rmse = evaluate_protein_prediction(groundtruth, pred)

In [40]:
rmse.mean(), pearson.mean()

(np.float64(0.9008203660374257), np.float64(0.5524379720805889))